# Robustness Checks Notebook

This notebook validates whether conclusions remain stable under alternative specifications and samples.

Checks:
- Ramsey RESET on forecast calibration regressions.
- Validation-vs-test error stability tests.
- Tail-event exclusion sensitivity for DM tests.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats

if os.path.basename(os.getcwd()) == 'khanh_model_analysis':
    os.chdir('..')

sys.path.append('src')

from statistical_validation import (
    set_reproducible_seed,
    load_config,
    discover_forecasts,
    reset_specification_test,
    run_dm_comparisons,
    build_default_dm_pairs,
)

set_reproducible_seed(42)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_columns', 200)

#### Interpretation
This setup cell prepares robustness utilities. Its role is to ensure all follow-up tests are run under the same package versions and helper definitions.

In [2]:
config = load_config('configs/pipeline_config.yaml')
active_target = config['active_target']
forecasts = discover_forecasts(config)

print(f'Active target: {active_target}')
print('Rows:', len(forecasts))
print('Models:', sorted(forecasts['Model'].unique().tolist()))

Active target: PHP
Rows: 38070
Models: ['arima', 'baseline_ar1', 'baseline_mean', 'baseline_rw', 'hybrid_arima_mlp', 'hybrid_arima_svr', 'hybrid_var_mlp', 'hybrid_var_svr', 'var']


#### Interpretation
The printed metadata verifies active target, sample size, and model count. These checks protect robustness conclusions from accidental data-path mismatch.

### Why robustness matters

A model can look strong on one split and still fail under small specification changes. The checks below ask whether conclusions survive a RESET test, whether validation errors behave similarly on the test set, and whether DM conclusions are driven by a few extreme tail observations. That is the right standard for a forecasting pipeline that may later expand to ARIMAX, VARX, or new hybrid variants.

## 1) Ramsey RESET (Specification Bias)

Null: no omitted non-linear terms in calibration equation `Actual ~ Forecast`.

In [3]:
reset_val = reset_specification_test(forecasts, set_name='val')
reset_test = reset_specification_test(forecasts, set_name='test')

display(reset_val.sort_values(['Pair', 'RESET_pvalue']))
display(reset_test.sort_values(['Pair', 'RESET_pvalue']))

reset_compare = reset_val.merge(
    reset_test,
    on=['Pair', 'Model'],
    suffixes=('_val', '_test'),
)
reset_compare['stable_conclusion'] = (
    reset_compare['Specification_bias_5pct_val'] == reset_compare['Specification_bias_5pct_test']
)
display(reset_compare[['Pair', 'Model', 'RESET_pvalue_val', 'RESET_pvalue_test', 'stable_conclusion']])

,Pair,Model,Set,R2,RESET_F,RESET_pvalue,Specification_bias_5pct,RESET_note
4,CNYPHP_RET,hybrid_arima_mlp,val,0.088081,22.879028,0.000002,True,OK
5,CNYPHP_RET,hybrid_arima_svr,val,0.017301,21.269198,0.000005,True,OK
0,CNYPHP_RET,arima,val,0.099395,9.426465,0.002278,True,OK
1,CNYPHP_RET,baseline_ar1,val,0.082906,7.960822,0.005007,True,OK
8,CNYPHP_RET,var,val,0.003700,2.973281,0.085386,False,OK
7,CNYPHP_RET,hybrid_var_svr,val,0.001677,1.693737,0.193822,False,OK
6,CNYPHP_RET,hybrid_var_mlp,val,0.007336,0.189341,0.663689,False,OK
2,CNYPHP_RET,baseline_mean,val,NaN,NaN,NaN,False,"Skipped: forecast is constant, RESET not defined."
3,CNYPHP_RET,baseline_rw,val,NaN,NaN,NaN,False,"Skipped: forecast is constant, RESET not defined."
17,HKDPHP_RET,var,val,0.008635,4.765699,0.029585,True,OK


,Pair,Model,Set,R2,RESET_F,RESET_pvalue,Specification_bias_5pct,RESET_note
8,CNYPHP_RET,var,test,0.128851,55.403649,5.604828e-13,True,OK
1,CNYPHP_RET,baseline_ar1,test,0.114140,43.764347,1.130262e-10,True,OK
0,CNYPHP_RET,arima,test,0.148077,39.980309,6.567532e-10,True,OK
6,CNYPHP_RET,hybrid_var_mlp,test,0.207689,21.214998,5.450566e-06,True,OK
7,CNYPHP_RET,hybrid_var_svr,test,0.186407,10.166515,1.537193e-03,True,OK
4,CNYPHP_RET,hybrid_arima_mlp,test,0.237579,4.349233,3.762875e-02,True,OK
5,CNYPHP_RET,hybrid_arima_svr,test,0.204722,0.134507,7.139883e-01,False,OK
2,CNYPHP_RET,baseline_mean,test,NaN,NaN,NaN,False,"Skipped: forecast is constant, RESET not defined."
3,CNYPHP_RET,baseline_rw,test,NaN,NaN,NaN,False,"Skipped: forecast is constant, RESET not defined."
17,HKDPHP_RET,var,test,0.147378,53.119061,1.569226e-12,True,OK


,Pair,Model,RESET_pvalue_val,RESET_pvalue_test,stable_conclusion
0,CNYPHP_RET,arima,0.002278,6.567532e-10,True
1,CNYPHP_RET,baseline_ar1,0.005007,1.130262e-10,True
2,CNYPHP_RET,baseline_mean,NaN,NaN,True
3,CNYPHP_RET,baseline_rw,NaN,NaN,True
4,CNYPHP_RET,hybrid_arima_mlp,0.000002,3.762875e-02,True
5,CNYPHP_RET,hybrid_arima_svr,0.000005,7.139883e-01,False
6,CNYPHP_RET,hybrid_var_mlp,0.663689,5.450566e-06,False
7,CNYPHP_RET,hybrid_var_svr,0.193822,1.537193e-03,False
8,CNYPHP_RET,var,0.085386,5.604828e-13,False
9,HKDPHP_RET,arima,0.418487,1.531152e-11,False


#### Interpretation
RESET results test functional-form adequacy. Rejection implies potential misspecification and motivates flexible nonlinear components; non-rejection supports the baseline functional form.

## 2) Validation vs Test Error Stability

A robust model should not deteriorate sharply when moving from validation to test.

In [4]:
stability_rows = []
for (pair, model), chunk in forecasts.groupby(['Pair', 'Model']):
    val_abs = chunk.loc[chunk['Set'] == 'val', 'AE'].dropna().values
    test_abs = chunk.loc[chunk['Set'] == 'test', 'AE'].dropna().values
    if len(val_abs) < 10 or len(test_abs) < 10:
        continue

    mw = stats.mannwhitneyu(val_abs, test_abs, alternative='two-sided')
    stability_rows.append({
        'Pair': pair,
        'Model': model,
        'Val_MAE': float(np.mean(val_abs)),
        'Test_MAE': float(np.mean(test_abs)),
        'Delta_Test_minus_Val': float(np.mean(test_abs) - np.mean(val_abs)),
        'MW_pvalue': float(mw.pvalue),
        'Distribution_shift_5pct': float(mw.pvalue) < 0.05,
    })

stability_df = pd.DataFrame(stability_rows).sort_values(['Pair', 'Delta_Test_minus_Val'])
display(stability_df)

,Pair,Model,Val_MAE,Test_MAE,Delta_Test_minus_Val,MW_pvalue,Distribution_shift_5pct
7,CNYPHP_RET,hybrid_var_svr,0.425791,0.393824,-0.031967,0.476342,False
5,CNYPHP_RET,hybrid_arima_svr,0.407449,0.382081,-0.025368,0.312135,False
6,CNYPHP_RET,hybrid_var_mlp,0.409594,0.387458,-0.022135,0.573202,False
4,CNYPHP_RET,hybrid_arima_mlp,0.396912,0.377803,-0.019109,0.291328,False
8,CNYPHP_RET,var,0.404971,0.394842,-0.010129,0.790956,False
2,CNYPHP_RET,baseline_mean,0.404679,0.400646,-0.004033,0.334038,False
3,CNYPHP_RET,baseline_rw,0.404617,0.400792,-0.003825,0.348599,False
0,CNYPHP_RET,arima,0.392797,0.389092,-0.003705,0.551186,False
1,CNYPHP_RET,baseline_ar1,0.396941,0.395346,-0.001595,0.571670,False
16,HKDPHP_RET,hybrid_var_svr,0.301915,0.376444,0.074529,0.004180,True


#### Interpretation
Validation-versus-test stability compares model behavior across splits. Small drift in metrics indicates robust generalization; large drift suggests overfitting or regime sensitivity.

## 3) Tail-Event Exclusion Sensitivity (DM)

We remove the top 1% absolute forecast errors by pair/model and re-run DM tests to verify conclusions are not driven only by a few extreme events.

In [5]:
trimmed = forecasts.copy()
trimmed['keep'] = True

for (pair, model), chunk in trimmed.groupby(['Pair', 'Model']):
    q = chunk['AE'].quantile(0.99)
    idx = chunk[chunk['AE'] > q].index
    trimmed.loc[idx, 'keep'] = False

trimmed = trimmed[trimmed['keep']].drop(columns=['keep'])

models = sorted(trimmed['Model'].unique().tolist())
dm_pairs = build_default_dm_pairs(models)

full_dm = run_dm_comparisons(forecasts, dm_pairs, criterion='mse', set_name='test')
trim_dm = run_dm_comparisons(trimmed, dm_pairs, criterion='mse', set_name='test')

merged_dm = full_dm.merge(
    trim_dm,
    on=['Pair', 'Model_A', 'Model_B', 'Loss', 'Set'],
    suffixes=('_full', '_trim')
)
merged_dm['same_significance'] = (
    (merged_dm['p_value_full'] < 0.05) == (merged_dm['p_value_trim'] < 0.05)
)
display(merged_dm.sort_values(['Pair', 'p_value_trim']))

,Pair,Model_A,Model_B,Loss,Set,DM_stat_full,p_value_full,n_obs_full,A_better_than_B_5pct_full,DM_stat_trim,p_value_trim,n_obs_trim,A_better_than_B_5pct_trim,same_significance
4,CNYPHP_RET,arima,baseline_ar1,mse,test,-2.916155,0.003733,423,True,-2.320902,0.020776,416,True,True
6,CNYPHP_RET,arima,baseline_rw,mse,test,-2.587261,0.010008,423,True,-2.293753,0.022304,416,True,True
5,CNYPHP_RET,arima,baseline_mean,mse,test,-2.597348,0.009723,423,True,-2.290644,0.022485,416,True,True
9,CNYPHP_RET,hybrid_arima_mlp,baseline_rw,mse,test,-2.118611,0.034707,423,True,-2.091072,0.037129,416,True,True
8,CNYPHP_RET,hybrid_arima_mlp,baseline_mean,mse,test,-2.121856,0.034432,423,True,-2.089852,0.037239,416,True,True
7,CNYPHP_RET,hybrid_arima_mlp,baseline_ar1,mse,test,-1.889760,0.059475,423,False,-1.873852,0.061653,416,False,True
21,CNYPHP_RET,var,baseline_rw,mse,test,-2.187641,0.029244,423,True,-1.602262,0.109857,417,False,False
20,CNYPHP_RET,var,baseline_mean,mse,test,-2.194668,0.028732,423,True,-1.599813,0.110399,417,False,False
12,CNYPHP_RET,hybrid_arima_svr,baseline_rw,mse,test,-1.622145,0.105519,423,False,-1.581081,0.114621,416,False,True
11,CNYPHP_RET,hybrid_arima_svr,baseline_mean,mse,test,-1.623035,0.105329,423,False,-1.578820,0.115139,416,False,True


#### Interpretation
Tail-sensitivity DM analysis checks whether conclusions hold after trimming extremes. If significance patterns persist, relative model ranking is not driven only by outlier episodes.

In [6]:
out_dir = f'results/{active_target}/evaluation'
os.makedirs(out_dir, exist_ok=True)

reset_test.to_csv(f'{out_dir}/reset_test_results.csv', index=False)
stability_df.to_csv(f'{out_dir}/val_test_error_stability.csv', index=False)
merged_dm.to_csv(f'{out_dir}/dm_tail_sensitivity.csv', index=False)

print('Saved robustness outputs to', out_dir)

Saved robustness outputs to results/PHP/evaluation


#### Interpretation
This save step records robustness tables for reproducible reporting. Stored outputs should match the tables displayed above to ensure consistency.